# Modin_Joblib - Small Dataset

**Summary of this notebook**

This notebook presents a benchmarking analysis of common data processing operations using Joblib and Modin on the **small dataset**. The tasks include filtering, arithmetic operations, aggregations, groupbys, joins, and profiling with cProfile, allowing a performance comparison between the two libraries.

**Setup and configurations**

In [1]:
# Importing required packages
import pandas as pd
import pyspark
import pyarrow
import numpy as np
import os
import requests
import time
import joblib
import modin.pandas as md
from dask.distributed import Client, LocalCluster
import modin.config as cfg
from joblib import parallel_backend, Parallel, delayed
from joblib import Parallel, delayed, dump, load
import dask.dataframe as dd

# Check versions
print("pandas version:", pd.__version__)
print("pyspark version:", pyspark.__version__)
print("pyarrow version:", pyarrow.__version__)
print("numpy version:", np.__version__)
print("modin version:", md.__version__)
print("joblib version:", joblib.__version__)

pandas version: 1.3.5
pyspark version: 3.4.4
pyarrow version: 1.0.1
numpy version: 1.19.5
modin version: 0.12.1
joblib version: 1.3.2


In [ ]:
#Supress warnings to reduce output clutter
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

Modin was configured to use Dask as its execution engine, and a Dask client was started. After that, the dataset was read with Modin and basic information about the DataFrame was displayed.

In [ ]:
# Configure Modin to use Dask
cfg.Engine.put("dask")

In [5]:
client = Client()

In [6]:
modin_data = md.read_parquet("Samples/sample_2009_2010_small_clean.parquet")

In [7]:
modin_data.info()

<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 730009 entries, 0 to 730008
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ---------------------  ---------------  -----         
 0   index                  730009 non-null  int64
 1   vendor_name            730009 non-null  object
 2   Trip_Pickup_DateTime   730009 non-null  datetime64[ns]
 3   Trip_Dropoff_DateTime  730009 non-null  object
 4   Passenger_Count        730009 non-null  int64
 5   Trip_Distance          730009 non-null  float64
 6   Start_Lon              730009 non-null  float64
 7   Start_Lat              730009 non-null  float64
 8   End_Lon                730009 non-null  float64
 9   End_Lat                730009 non-null  float64
 10  Payment_Type           730009 non-null  object
 11  Fare_Amt               730009 non-null  float64
 12  surcharge              730009 non-null  float64
 13  Tip_Amt                730009 non-null  float64
 14  Tolls_Amt        

# Benchmark

# Modin

Several benchmark functions were defined to measure execution time of different operations using Modin. Then, benchmark results are stored in dictionaries for standard, filtered, and cached scenarios.

In [8]:
def benchmark(f, df, benchmarks, name, **kwargs):
    """Benchmark the given function against the given DataFrame.

    Parameters
    ----------
    f: function to benchmark
    df: data frame
    benchmarks: container for benchmark results
    name: task name

    Returns
    -------
    Duration (in seconds, high precision) of the given operation
    """
    start_time = time.perf_counter()  # alta precisão
    ret = f(df, **kwargs)
    duration = time.perf_counter() - start_time
    benchmarks['duration'].append(duration)
    benchmarks['task'].append(name)
    print(f"{name} took: {duration:.6f} seconds")
    return duration

def get_results(benchmarks):
    """Return a pandas DataFrame containing benchmark results."""
    return pd.DataFrame.from_dict(benchmarks)

In [9]:
def read_file_parquet(df=None):
    return md.read_parquet("Samples/sample_2009_2010_small_clean.parquet")

def count(df=None):
    return len(df)

def count_index_length(df=None):
    return len(df.index)

def mean(df):
    return df.Fare_Amt.mean()

def standard_deviation(df):
    return df.Fare_Amt.std()

def mean_of_sum(df):
    return (df.Fare_Amt + df.Tip_Amt).mean()

def sum_columns(df):
    x = df.Fare_Amt + df.Tip_Amt

    return x

def mean_of_product(df):
    return (df.Fare_Amt * df.Tip_Amt).mean()

def product_columns(df):
    x = df.Fare_Amt * df.Tip_Amt

    return x

def value_counts(df):
    val_counts = df.Fare_Amt.value_counts()
    return val_counts

def complicated_arithmetic_operation(df):
    theta_1 = df.Start_Lon
    phi_1 = df.Start_Lat
    theta_2 = df.End_Lon
    phi_2 = df.End_Lat
    temp = (np.sin((theta_2 - theta_1) / 2 * np.pi / 180) ** 2
           + np.cos(theta_1 * np.pi / 180) * np.cos(theta_2 * np.pi / 180) * np.sin((phi_2 - phi_1) / 2 * np.pi / 180) ** 2)
    ret = np.multiply(np.arctan2(np.sqrt(temp), np.sqrt(1-temp)),2)

    return ret

def mean_of_complicated_arithmetic_operation(df):
    theta_1 = df.Start_Lon
    phi_1 = df.Start_Lat
    theta_2 = df.End_Lon
    phi_2 = df.End_Lat
    temp = (np.sin((theta_2 - theta_1) / 2 * np.pi / 180) ** 2
           + np.cos(theta_1 * np.pi / 180) * np.cos(theta_2 * np.pi / 180) * np.sin((phi_2 - phi_1) / 2 * np.pi / 180) ** 2)
    ret = np.multiply(np.arctan2(np.sqrt(temp), np.sqrt(1-temp)),2)
    return ret.mean()

def groupby_statistics(df):
    gb = df.groupby(by='Passenger_Count').agg(
      {
        'Fare_Amt': ['mean', 'std'],
        'Tip_Amt': ['mean', 'std']
      }
    )
    return gb

other = md.DataFrame(groupby_statistics(modin_data))
other.columns = md.Index([e[0]+'_' + e[1] for e in other.columns.tolist()])

def join_count(df, other):
    return len(df.merge(other, left_index=True, right_index=True))

def join_data(df, other):
    return df.merge(other, left_index=True, right_index=True)


In [10]:
modin_benchmarks = {
    'duration': [],  # in seconds
    'task': [],
}

modin_benchmarks_filtered = {
    'duration': [],  # in seconds
    'task': [],
}

modin_benchmarks_cache = {
    'duration': [],  # in seconds
    'task': [],
}

After that, each function was benchmarked using the benchmark() utility. The tasks were applied to the Modin DataFrame.

In [ ]:
# Execute benchmark
benchmark(read_file_parquet, df=None, benchmarks=modin_benchmarks, name='read file')
benchmark(count, df=modin_data, benchmarks=modin_benchmarks, name='count')
benchmark(count_index_length, df=modin_data, benchmarks=modin_benchmarks, name='count index length')
benchmark(mean, df=modin_data, benchmarks=modin_benchmarks, name='mean')
benchmark(standard_deviation, df=modin_data, benchmarks=modin_benchmarks, name='standard deviation')
benchmark(mean_of_sum, df=modin_data, benchmarks=modin_benchmarks, name='mean of columns addition')
benchmark(sum_columns, df=modin_data, benchmarks=modin_benchmarks, name='addition of columns')
benchmark(mean_of_product, df=modin_data, benchmarks=modin_benchmarks, name='mean of columns multiplication')
benchmark(product_columns, df=modin_data, benchmarks=modin_benchmarks, name='multiplication of columns')
benchmark(value_counts, df=modin_data, benchmarks=modin_benchmarks, name='value counts')
benchmark(complicated_arithmetic_operation, df=modin_data, benchmarks=modin_benchmarks, name='complex arithmetic ops')
benchmark(mean_of_complicated_arithmetic_operation, df=modin_data, benchmarks=modin_benchmarks, name='mean of complex arithmetic ops')
benchmark(groupby_statistics, df=modin_data, benchmarks=modin_benchmarks, name='groupby statistics')
benchmark(join_count, modin_data, benchmarks=modin_benchmarks, name='join count', other=other)
benchmark(join_data, modin_data, benchmarks=modin_benchmarks, name='join', other=other)

read file took: 0.417695 seconds
count took: 0.000019 seconds
count index length took: 0.000005 seconds
mean took: 0.107752 seconds
standard deviation took: 0.095688 seconds
mean of columns addition took: 0.456715 seconds
addition of columns took: 0.341149 seconds
mean of columns multiplication took: 0.479356 seconds
multiplication of columns took: 0.320562 seconds
value counts took: 0.496683 seconds
complex arithmetic ops took: 3.758042 seconds
mean of complex arithmetic ops took: 3.952722 seconds
groupby statistics took: 0.263856 seconds
join count took: 0.391485 seconds
join took: 0.360196 seconds


0.3601963800001613

In [ ]:
# Retrieve benchmark results and remove duplicate task entries
modin_res_temp = get_results(modin_benchmarks).drop_duplicates(subset='task').set_index('task')

# Print the result DataFrame
print(modin_res_temp)

# Define the file path for saving
filename = 'Results_Modin/modin_local_small'

# Save the result DataFrame as a Parquet file
modin_res_temp.to_parquet(filename)

print(f'O arquivo Parquet foi salvo em {filename}.')

                                duration
task                                    
read file                       0.417695
count                           0.000019
count index length              0.000005
mean                            0.107752
standard deviation              0.095688
mean of columns addition        0.456715
addition of columns             0.341149
mean of columns multiplication  0.479356
multiplication of columns       0.320562
value counts                    0.496683
complex arithmetic ops          3.758042
mean of complex arithmetic ops  3.952722
groupby statistics              0.263856
join count                      0.391485
join                            0.360196
O arquivo Parquet foi salvo em Results_Modin/modin_local_small.


## cProfile

In this section, the same Modin benchmarks were executed using the cProfile module to collect detailed profiling statistics. This allows for deeper analysis of function performance by identifying the most time-consuming operations. Execution times and profiling data were saved for later comparison.

In [13]:
import cProfile
import matplotlib.pyplot as plt
from io import StringIO
import pstats

In [ ]:
def benchmark_cprofile(f, df, benchmarks, name, profile_data=None, **kwargs):
    """
    Benchmark and profile the execution of a function using cProfile (without file I/O).
    
    Parameters:
    - f: function to benchmark
    - df: input dataframe (or None)
    - benchmarks: dictionary to store timing results
    - name: name of the benchmark task
    - profile_data: optional list to store detailed profiling data
    - kwargs: other keyword arguments passed to the function
    """
    pr = cProfile.Profile() #Create a profile object
    pr.enable()             #Start profiling

    start_time = time.perf_counter()    
    ret = f(df, **kwargs)
    duration = time.perf_counter() - start_time

    pr.disable()

    benchmarks['duration'].append(duration)
    benchmarks['task'].append(name)
    print(f"{name} took: {duration:.4f} seconds")

    # Print top 15 most time-consuming operations to the terminal    
    ps = pstats.Stats(pr)
    ps.sort_stats(pstats.SortKey.CUMULATIVE)
    ps.print_stats(15)

    # Store detailed profiling data for later visualization
    if profile_data is not None:
        stats_data = []
        for func, stat in ps.stats.items():
            filename, lineno, function = func
            ncalls, _, percall, cumtime, _ = stat
            stats_data.append({
                'task': name,
                'function': f"{filename}:{lineno} ({function})",
                'percall': percall,
                'cumtime': cumtime,
                'ncalls': ncalls
            })
        profile_data.append(pd.DataFrame(stats_data))

    return duration

In [ ]:
# Store benchmark durations and task names
modin_benchmarks_cprofile = {'duration': [], 'task': []}
# Store profiling details per function
modin_profile_data = []

# Run the benchmarks with profiling enabled
benchmark_cprofile(read_file_parquet, df=None, benchmarks=modin_benchmarks_cprofile, name='read file', profile_data=modin_profile_data)
benchmark_cprofile(count, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='count', profile_data=modin_profile_data)
benchmark_cprofile(count_index_length, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='count index length', profile_data=modin_profile_data)
benchmark_cprofile(mean, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='mean', profile_data=modin_profile_data)
benchmark_cprofile(standard_deviation, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='standard deviation', profile_data=modin_profile_data)
benchmark_cprofile(mean_of_sum, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='mean of columns addition', profile_data=modin_profile_data)
benchmark_cprofile(sum_columns, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='addition of columns', profile_data=modin_profile_data)
benchmark_cprofile(mean_of_product, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='mean of columns multiplication', profile_data=modin_profile_data)
benchmark_cprofile(product_columns, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='multiplication of columns', profile_data=modin_profile_data)
benchmark_cprofile(value_counts, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='value counts', profile_data=modin_profile_data)
benchmark_cprofile(complicated_arithmetic_operation, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='complex arithmetic ops', profile_data=modin_profile_data)
benchmark_cprofile(mean_of_complicated_arithmetic_operation, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='mean of complex arithmetic ops', profile_data=modin_profile_data)
benchmark_cprofile(groupby_statistics, df=modin_data, benchmarks=modin_benchmarks_cprofile, name='groupby statistics', profile_data=modin_profile_data)
benchmark_cprofile(join_count, modin_data, benchmarks=modin_benchmarks_cprofile, name='join count', other=other, profile_data=modin_profile_data)
benchmark_cprofile(join_data, modin_data, benchmarks=modin_benchmarks_cprofile, name='join', other=other, profile_data=modin_profile_data)

read file took: 0.3907 seconds
         63912 function calls (60763 primitive calls) in 0.391 seconds

   Ordered by: cumulative time
   List reduced from 526 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.391    0.391 /var/tmp/ipykernel_10132/404667543.py:1(read_file_parquet)
        1    0.000    0.000    0.391    0.391 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/pandas/io.py:203(read_parquet)
        1    0.000    0.000    0.391    0.391 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/execution/dispatching/factories/dispatcher.py:172(read_parquet)
        1    0.000    0.000    0.391    0.391 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/execution/dispatching/factories/factories.py:189(_read_parquet)
        1    0.000    0.000    0.391    0.391 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/io/file_dispatcher.py:128(read)
        1    0.000    0

0.4404928660001133

In [ ]:
# Concatenate all DataFrames
df_profile_all = pd.concat(modin_profile_data, ignore_index=True)

**NOTE:**

Instead of generating plots to visualize the profiling results, we opted to use the print function, as rendering multiple plots made the notebook too heavy and unresponsive when running in Google Cloud Platform (GCP).

In this way, the print_profile_per_task function provides a textual summary of the slowest internal functions (based on profiling) for each benchmarked operation, directly printed to the console instead of generating visual plots.

The printed output includes:

- A separate section for each benchmarked operation;
- For each operation, the top N internal functions (based on profiling metrics) are listed in descending order of execution time.
- The two available profiling metrics are:
  - 'percall': Average time per individual call to the function (in seconds).

  - 'cumtime': Total cumulative time spent in the function, including all nested calls.

- For each listed function, the name, file, and line number are shown alongside the corresponding metric value.

This function is especially useful when working in resource-constrained environments like Google Cloud Platform (GCP), where generating heavy visual plots can make notebooks sluggish or unusable. Instead, print_profile_per_task provides a lightweight and effective way to analyze performance bottlenecks.

In [ ]:
def print_profile_per_task(profile_data_list, metric='percall', top_n=10):
    """
    Prints the top slowest functions (by percall or cumtime) for each benchmark task.
    
    Parameters:
    - profile_data_list: list of DataFrames (one per benchmark task), each containing profiling data
    - metric: the metric to sort by ('percall' or 'cumtime')
    - top_n: number of slowest functions to display per task
    """
    for df in profile_data_list:
        task_name = df['task'].iloc[0]   # All rows in this DataFrame refer to the same task
        df_sorted = df.sort_values(by=metric, ascending=False).head(top_n)

        print(f"\n{'='*50}")
        print(f"Tarefa: {task_name}")
        print(f"Top {top_n} funções por {metric}:\n")
        print(df_sorted[['function', metric]].to_string(index=False))

In [18]:
print_profile_per_task(modin_profile_data, metric='percall', top_n=25)


Tarefa: read file
Top 25 funções por percall:

                                                                                     function  percall
                                           ~:0 (<method 'acquire' of '_thread.lock' objects>) 0.351934
                                      /opt/conda/envs/cdle/lib/python3.7/pickle.py:485 (save) 0.003024
                               /opt/conda/envs/cdle/lib/python3.7/tokenize.py:487 (_tokenize) 0.002100
                /opt/conda/envs/cdle/lib/python3.7/collections/__init__.py:930 (__contains__) 0.001869
         /opt/conda/envs/cdle/lib/python3.7/site-packages/distributed/client.py:1628 (submit) 0.001700
                                   /opt/conda/envs/cdle/lib/python3.7/pickle.py:441 (memoize) 0.001661
                                  /opt/conda/envs/cdle/lib/python3.7/pickle.py:738 (save_str) 0.001498
                                                           ~:0 (<built-in method posix.stat>) 0.001464
                         

**Operations with filtering**

The **filtered benchmarks** focus on evaluating performance after applying a data filtering operation.

The dataset is filtered to include only rows where the Tip_Amt is between 1 and 5 (inclusive), simulating a common preprocessing step.

This filtered DataFrame is then passed through the same benchmarking pipeline as before, testing various operations like count, mean, groupby, and joins.

This allows us to compare the performance of Modin on both raw and pre-filtered data, which is often more representative of real-world data workflows.

In [ ]:
expr_filter = 'Tip_Amt >= 1 and Tip_Amt <= 5'
filtered = modin_data.query(expr_filter)

if not isinstance(filtered, (pd.DataFrame, md.DataFrame)):
    filtered = md.DataFrame(filtered) if isinstance(data_filtered, md.DataFrame) else pd.DataFrame(filtered)

modin_filtered = filtered

In [ ]:
benchmark(read_file_parquet, df=None, benchmarks=modin_benchmarks_filtered, name='read file')
benchmark(count, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='count')
benchmark(count_index_length, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='count index length')
benchmark(mean, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='mean')
benchmark(standard_deviation, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='standard deviation')
benchmark(mean_of_sum, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='mean of columns addition')
benchmark(sum_columns, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='addition of columns')
benchmark(mean_of_product, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='mean of columns multiplication')
benchmark(product_columns, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='multiplication of columns')
benchmark(value_counts, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='value counts')
benchmark(complicated_arithmetic_operation, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='complex arithmetic ops')
benchmark(mean_of_complicated_arithmetic_operation, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='mean of complex arithmetic ops')
benchmark(groupby_statistics, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='groupby statistics')
benchmark(join_count, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='join count', other=other)
benchmark(join_data, df=modin_filtered, benchmarks=modin_benchmarks_filtered, name='join', other=other)

read file took: 0.359517 seconds
count took: 0.000018 seconds
count index length took: 0.000006 seconds
mean took: 0.105271 seconds
standard deviation took: 0.103087 seconds
mean of columns addition took: 0.417071 seconds
addition of columns took: 0.330940 seconds
mean of columns multiplication took: 0.435747 seconds
multiplication of columns took: 0.329306 seconds
value counts took: 0.597272 seconds
complex arithmetic ops took: 3.651512 seconds
mean of complex arithmetic ops took: 3.667286 seconds
groupby statistics took: 0.249523 seconds
join count took: 0.342852 seconds
join took: 0.324764 seconds


0.32476380100001734

**Save results**

In [28]:
modin_res_filtered = get_results(modin_benchmarks_filtered).drop_duplicates(subset='task').set_index('task')
print(modin_res_filtered)

                                duration
task                                    
read file                       0.396246
count                           0.000015
count index length              0.000004
mean                            0.119379
standard deviation              0.119718
mean of columns addition        0.456034
addition of columns             0.333659
mean of columns multiplication  0.451155
multiplication of columns       0.346534
value counts                    0.633684
complex arithmetic ops          3.651512
mean of complex arithmetic ops  3.667286
groupby statistics              0.249523
join count                      0.342852
join                            0.324764


In [ ]:
filename_filter = 'Results_Modin/modin_filtered_local_small'
modin_res_filtered.to_parquet(filename_filter)
print(f'O arquivo Parquet foi salvo em {filename_filter}.')

O arquivo Parquet foi salvo em Results_Modin/modin_filtered_local_small.


**Operations with filtering using cProfile**

The filtered cProfile benchmarks collect detailed performance insights for each filtered operation using the cProfile module.

- Each operation (e.g., count, mean, groupby) is executed on the previously filtered dataset, and function-level profiling data is collected.

- The profiling results are stored in a list of DataFrames, one per benchmarked task, capturing execution time per function call.

- Once again, since graphical profiling plots were too resource-intensive for GCP notebooks, the top 25 slowest functions per operation are printed to screen using print_profile_per_task, based on average time per call (percall).

In [ ]:
modin_benchmarks_filtered_cprofile = {'duration': [], 'task': []}
modin_filtered_profile_data = []

In [ ]:
# Execute benchmarks with cProfile
benchmark_cprofile(read_file_parquet, df=None, benchmarks=modin_benchmarks_filtered_cprofile, name='read file', profile_data=modin_filtered_profile_data)
benchmark_cprofile(count, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='count', profile_data=modin_filtered_profile_data)
benchmark_cprofile(count_index_length, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='count index length', profile_data=modin_filtered_profile_data)
benchmark_cprofile(mean, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='mean', profile_data=modin_filtered_profile_data)
benchmark_cprofile(standard_deviation, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='standard deviation', profile_data=modin_filtered_profile_data)
benchmark_cprofile(mean_of_sum, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='mean of columns addition', profile_data=modin_filtered_profile_data)
benchmark_cprofile(sum_columns, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='addition of columns', profile_data=modin_filtered_profile_data)
benchmark_cprofile(mean_of_product, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='mean of columns multiplication', profile_data=modin_filtered_profile_data)
benchmark_cprofile(product_columns, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='multiplication of columns', profile_data=modin_filtered_profile_data)
benchmark_cprofile(value_counts, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='value counts', profile_data=modin_filtered_profile_data)
benchmark_cprofile(complicated_arithmetic_operation, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='complex arithmetic ops', profile_data=modin_filtered_profile_data)
benchmark_cprofile(mean_of_complicated_arithmetic_operation, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='mean of complex arithmetic ops', profile_data=modin_filtered_profile_data)
benchmark_cprofile(groupby_statistics, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='groupby statistics', profile_data=modin_filtered_profile_data)
benchmark_cprofile(join_count, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='join count', other=other, profile_data=modin_filtered_profile_data)
benchmark_cprofile(join_data, df=modin_filtered, benchmarks=modin_benchmarks_filtered_cprofile, name='join', other=other, profile_data=modin_filtered_profile_data)

read file took: 0.3740 seconds
         63917 function calls (60768 primitive calls) in 0.374 seconds

   Ordered by: cumulative time
   List reduced from 526 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.374    0.374 /var/tmp/ipykernel_10132/404667543.py:1(read_file_parquet)
        1    0.000    0.000    0.374    0.374 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/pandas/io.py:203(read_parquet)
        1    0.000    0.000    0.374    0.374 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/execution/dispatching/factories/dispatcher.py:172(read_parquet)
        1    0.000    0.000    0.374    0.374 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/execution/dispatching/factories/factories.py:189(_read_parquet)
        1    0.000    0.000    0.374    0.374 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/io/file_dispatcher.py:128(read)
        1    0.000    0

0.4389695740001116

In [ ]:
# Concatenate all results
df_filtered_profile_all = pd.concat(modin_filtered_profile_data, ignore_index=True)

In [ ]:
print_profile_per_task(modin_filtered_profile_data, metric='percall', top_n=25)


Tarefa: read file
Top 25 funções por percall:

                                                                    function  percall
                          ~:0 (<method 'acquire' of '_thread.lock' objects>) 0.334186
                                          ~:0 (<built-in method posix.stat>) 0.002932
                     /opt/conda/envs/cdle/lib/python3.7/pickle.py:485 (save) 0.002924
              /opt/conda/envs/cdle/lib/python3.7/tokenize.py:487 (_tokenize) 0.002209
                           ~:0 (<method 'send' of '_socket.socket' objects>) 0.001728
                  /opt/conda/envs/cdle/lib/python3.7/pickle.py:441 (memoize) 0.001714
          ~:0 (<method 'get_file_info' of 'pyarrow._fs.FileSystem' objects>) 0.001503
                              ~:0 (<method 'match' of 're.Pattern' objects>) 0.001421
        ~:0 (<method 'finish' of 'pyarrow._dataset.DatasetFactory' objects>) 0.001312
                /opt/conda/envs/cdle/lib/python3.7/pickle.py:852 (save_dict) 0.001088
      

**Operations with cache**

In this section, the filtered Modin DataFrame was first converted to a Pandas DataFrame, then transformed into a Dask DataFrame with multiple partitions to leverage parallel processing and better memory management. 

The Dask DataFrame was persisted in memory to speed up repeated computations. Subsequently, a series of benchmark functions were executed on this cached data to measure performance across various operations. 

For join operations, the auxiliary dataset was also converted to Dask format to maintain consistency. 

Finally, the benchmark results were collected, deduplicated by task, saved to a Parquet file for future analysis, and printed.

In [ ]:
# Convert Modin DataFrame to Pandas DataFrame
dask_df = modin_filtered._to_pandas()
print("Conversão para Pandas realizada com sucesso.")

# Convert Pandas DataFrame to Dask DataFrame with 10 partitions
dask_df = dd.from_pandas(dask_df, npartitions=10)

# Persist the Dask DataFrame in memory for faster access
dask_df = client.persist(dask_df)
modin_cache = dask_df

Conversão para Pandas realizada com sucesso.


In [ ]:
# Run benchmarks on cached data
benchmark(read_file_parquet, df=None, benchmarks=modin_benchmarks_cache, name='read file')
benchmark(count, df=modin_cache, benchmarks=modin_benchmarks_cache, name='count')
benchmark(count_index_length, df=modin_cache, benchmarks=modin_benchmarks_cache, name='count index length')
benchmark(mean, df=modin_cache, benchmarks=modin_benchmarks_cache, name='mean')
benchmark(standard_deviation, df=modin_cache, benchmarks=modin_benchmarks_cache, name='standard deviation')
benchmark(mean_of_sum, df=modin_cache, benchmarks=modin_benchmarks_cache, name='mean of columns addition')
benchmark(sum_columns, df=modin_cache, benchmarks=modin_benchmarks_cache, name='addition of columns')
benchmark(mean_of_product, df=modin_cache, benchmarks=modin_benchmarks_cache, name='mean of columns multiplication')
benchmark(product_columns, df=modin_cache, benchmarks=modin_benchmarks_cache, name='multiplication of columns')
benchmark(value_counts, df=modin_cache, benchmarks=modin_benchmarks_cache, name='value counts')
benchmark(complicated_arithmetic_operation, df=modin_cache, benchmarks=modin_benchmarks_cache, name='complex arithmetic ops')
benchmark(groupby_statistics, df=modin_cache, benchmarks=modin_benchmarks_cache, name='groupby statistics')

# Prepare 'other' DataFrame for join operations, converting from Modin to Dask
other_dask = dd.from_pandas(other._to_pandas(), npartitions=10)  

benchmark(join_count, df=modin_cache, benchmarks=modin_benchmarks_cache, name='join count', other=other_dask)
benchmark(join_data, df=modin_cache, benchmarks=modin_benchmarks_cache, name='join', other=other_dask)

read file took: 0.413670 seconds
count took: 0.231584 seconds
count index length took: 0.026510 seconds
mean took: 0.003028 seconds
standard deviation took: 0.003417 seconds
mean of columns addition took: 0.002914 seconds
addition of columns took: 0.000664 seconds
mean of columns multiplication took: 0.003155 seconds
multiplication of columns took: 0.000639 seconds
value counts took: 0.001106 seconds
complex arithmetic ops took: 0.011581 seconds
groupby statistics took: 0.017277 seconds
join count took: 0.194686 seconds
join took: 0.010090 seconds


0.010090469000033409

In [ ]:
# Retrieve benchmark results, drop duplicate tasks and set index to 'task'
modin_res_cache = get_results(modin_benchmarks_cache).drop_duplicates(subset='task').set_index('task')
print(modin_res_cache)

                                duration
task                                    
read file                       0.413670
count                           0.231584
count index length              0.026510
mean                            0.003028
standard deviation              0.003417
mean of columns addition        0.002914
addition of columns             0.000664
mean of columns multiplication  0.003155
multiplication of columns       0.000639
value counts                    0.001106
complex arithmetic ops          0.011581
groupby statistics              0.017277
join count                      0.194686
join                            0.010090


In [ ]:
# Save results to Parquet file
filename_cache = 'Results_Modin/modin_cache_local_small'
modin_res_cache.to_parquet(filename_cache)
print(f'O arquivo Parquet foi salvo em {filename_cache}.')

O arquivo Parquet foi salvo em Results_Modin/modin_cache_local_small.


**Cache with cprofile**

In this section, performance profiling was conducted using cProfile for the previously cached Dask DataFrame (modin_cache). The goal was to capture fine-grained profiling data (function call durations and frequencies) across various common operations such as count, mean, groupby, and arithmetic operations. Each function was profiled individually using benchmark_cprofile, and the profiling data was stored in a list. At the end, all collected profiling results were concatenated into a single DataFrame and visualized using a custom print_profile_per_task function to highlight the top functions sorted by time per call (percall). Finally, the Dask client was restarted to clear memory and reset the environment.

This approach allows deeper analysis of function-level performance bottlenecks and behavior when working with cached data using Dask after converting from Modin.

In [ ]:
# === Benchmarks with cProfile for cached Modin data ===
modin_benchmarks_cache_cprofile = {'duration': [], 'task': []}
modin_cache_profile_data = []

In [ ]:
# Run benchmarks with cProfile
benchmark_cprofile(read_file_parquet, df=None, benchmarks=modin_benchmarks_cache_cprofile, name='read file', profile_data=modin_cache_profile_data)
benchmark_cprofile(count, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='count', profile_data=modin_cache_profile_data)
benchmark_cprofile(count_index_length, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='count index length', profile_data=modin_cache_profile_data)
benchmark_cprofile(mean, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='mean', profile_data=modin_cache_profile_data)
benchmark_cprofile(standard_deviation, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='standard deviation', profile_data=modin_cache_profile_data)
benchmark_cprofile(mean_of_sum, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='mean of columns addition', profile_data=modin_cache_profile_data)
benchmark_cprofile(sum_columns, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='addition of columns', profile_data=modin_cache_profile_data)
benchmark_cprofile(mean_of_product, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='mean of columns multiplication', profile_data=modin_cache_profile_data)
benchmark_cprofile(product_columns, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='multiplication of columns', profile_data=modin_cache_profile_data)
benchmark_cprofile(value_counts, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='value counts', profile_data=modin_cache_profile_data)
benchmark_cprofile(complicated_arithmetic_operation, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='complex arithmetic ops', profile_data=modin_cache_profile_data)
benchmark_cprofile(mean_of_complicated_arithmetic_operation, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='mean of complex arithmetic ops', profile_data=modin_cache_profile_data)
benchmark_cprofile(groupby_statistics, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='groupby statistics', profile_data=modin_cache_profile_data)
benchmark_cprofile(join_count, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='join count', other=other, profile_data=modin_cache_profile_data)
benchmark_cprofile(join_data, df=modin_cache, benchmarks=modin_benchmarks_cache_cprofile, name='join', other=other, profile_data=modin_cache_profile_data)

read file took: 0.3953 seconds
         63907 function calls (60759 primitive calls) in 0.395 seconds

   Ordered by: cumulative time
   List reduced from 526 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.395    0.395 /var/tmp/ipykernel_10132/404667543.py:1(read_file_parquet)
        1    0.000    0.000    0.395    0.395 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/pandas/io.py:203(read_parquet)
        1    0.000    0.000    0.395    0.395 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/execution/dispatching/factories/dispatcher.py:172(read_parquet)
        1    0.000    0.000    0.395    0.395 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/execution/dispatching/factories/factories.py:189(_read_parquet)
        1    0.000    0.000    0.395    0.395 /opt/conda/envs/cdle/lib/python3.7/site-packages/modin/core/io/file_dispatcher.py:128(read)
        1    0.000    0

0.3980278459998772

In [ ]:
# Concatenate all profiling results
df_cache_profile_all = pd.concat(modin_cache_profile_data, ignore_index=True)

In [ ]:
print_profile_per_task(modin_cache_profile_data, metric='percall', top_n=25)


Tarefa: read file
Top 25 funções por percall:

                                                                              function  percall
                                    ~:0 (<method 'acquire' of '_thread.lock' objects>) 0.355055
                               /opt/conda/envs/cdle/lib/python3.7/pickle.py:485 (save) 0.003963
                        /opt/conda/envs/cdle/lib/python3.7/tokenize.py:487 (_tokenize) 0.002143
                                                    ~:0 (<built-in method posix.stat>) 0.001940
                            /opt/conda/envs/cdle/lib/python3.7/pickle.py:441 (memoize) 0.001747
                           /opt/conda/envs/cdle/lib/python3.7/pickle.py:738 (save_str) 0.001687
  /opt/conda/envs/cdle/lib/python3.7/site-packages/distributed/client.py:1628 (submit) 0.001676
                  ~:0 (<method 'finish' of 'pyarrow._dataset.DatasetFactory' objects>) 0.001360
                                        ~:0 (<method 'match' of 're.Pattern' objects>) 0

In [ ]:
# Restart the Dask client to clear cluster state and free resources
client.restart()

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:45653/status,
Dashboard: http://127.0.0.1:45653/status,Workers: 4
Total threads: 8,Total memory: 31.34 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34143,Workers: 4
Dashboard: http://127.0.0.1:45653/status,Total threads: 8
Started: 10 minutes ago,Total memory: 31.34 GiB
Comm: tcp://127.0.0.1:37433,Total threads: 2
Dashboard: http://127.0.0.1:44295/status,Memory: 7.83 GiB
Nanny: tcp://127.0.0.1:32919,


# Joblib

This part of the notebook defines a set of benchmark functions to evaluate the performance of common data processing operations using Joblib. 

The dataset is loaded from a Parquet file and a benchmark_joblib function is also provided to run all tasks in parallel using Joblib and measure execution time. 

The goal is to compare execution speed across different backends or configurations.

In [43]:
joblib_data = pd.read_parquet("Samples/sample_2009_2010_small_clean.parquet")

In [ ]:
# Helper function to convert benchmark results into a pandas DataFrame
def get_results(benchmarks):
    """Returns a pandas DataFrame containing benchmark results"""
    return pd.DataFrame.from_dict(benchmarks)

def read_file_parquet(df=None):
    return pd.read_parquet("Samples/sample_2009_2010_small_clean.parquet")

def count(df=None):
    return len(df)

def count_index_length(df=None):
    return len(df.index)

def mean(df):
    return df.Fare_Amt.mean()

def standard_deviation(df):
    return df.Fare_Amt.std()

def mean_of_sum(df):
    return (df.Fare_Amt + df.Tip_Amt).mean()

def sum_columns(df):
    x = df.Fare_Amt + df.Tip_Amt

    return x

def mean_of_product(df):
    return (df.Fare_Amt * df.Tip_Amt).mean()

def product_columns(df):
    x = df.Fare_Amt * df.Tip_Amt

    return x

def value_counts(df):
    val_counts = df.Fare_Amt.value_counts()
    return val_counts

def complicated_arithmetic_operation(df):
    theta_1 = df.Start_Lon
    phi_1 = df.Start_Lat
    theta_2 = df.End_Lon
    phi_2 = df.End_Lat
    temp = (np.sin((theta_2 - theta_1) / 2 * np.pi / 180) ** 2
           + np.cos(theta_1 * np.pi / 180) * np.cos(theta_2 * np.pi / 180) * np.sin((phi_2 - phi_1) / 2 * np.pi / 180) ** 2)
    ret = np.multiply(np.arctan2(np.sqrt(temp), np.sqrt(1-temp)),2)

    return ret

def mean_of_complicated_arithmetic_operation(df):
    theta_1 = df.Start_Lon
    phi_1 = df.Start_Lat
    theta_2 = df.End_Lon
    phi_2 = df.End_Lat
    temp = (np.sin((theta_2 - theta_1) / 2 * np.pi / 180) ** 2
           + np.cos(theta_1 * np.pi / 180) * np.cos(theta_2 * np.pi / 180) * np.sin((phi_2 - phi_1) / 2 * np.pi / 180) ** 2)
    ret = np.multiply(np.arctan2(np.sqrt(temp), np.sqrt(1-temp)),2)
    return ret.mean()

def groupby_statistics(df):
    gb = df.groupby(by='Passenger_Count').agg(
      {
        'Fare_Amt': ['mean', 'std'],
        'Tip_Amt': ['mean', 'std']
      }
    )
    return gb

other = pd.DataFrame(groupby_statistics(joblib_data))
other.columns = pd.Index([e[0]+'_' + e[1] for e in other.columns.tolist()])

def join_count(df, other):
    return len(df.merge(other, left_index=True, right_index=True))

def join_data(df, other):
    return df.merge(other, left_index=True, right_index=True)

In [ ]:
# === Joblib benchmarking utility ===

# Run benchmark tasks in parallel using Joblib
def benchmark_joblib(tasks, n_jobs=-1, save_path=None):
    """
    Executes parallel benchmarks for a list of tasks and optionally saves the results.
    
    tasks: list of tuples (function, df, kwargs dict, task name)
    n_jobs: number of parallel jobs (-1 = use all cores)
    save_path: optional path to save results as a Parquet file
    
    Returns: DataFrame with 'task' as index and 'duration' in seconds
    """
    
    def run_task(f, df, kwargs, name):
        start = time.perf_counter()
        f(df, **kwargs)
        elapsed = time.perf_counter() - start
        return (name, elapsed)
    
    # Run all tasks in parallel
    results = Parallel(n_jobs=n_jobs)(
        delayed(run_task)(f, df, kwargs, name) for f, df, kwargs, name in tasks
    )
    
    # Create DataFrame and clean duplicates
    df_results = pd.DataFrame(results, columns=['task', 'duration']) \
                   .drop_duplicates(subset='task') \
                   .set_index('task')
    
    # Save to Parquet if specified
    if save_path:
        df_results.to_parquet(save_path)
        print(f'O arquivo Parquet foi salvo em {save_path}.')
    
    return df_results

Three dictionaries (joblib_benchmarks, joblib_benchmarks_filtered, joblib_benchmarks_cache) are initialized to store benchmark results. 

Then, a list of tasks is defined, where each task represents a function to benchmark (e.g., count, mean, groupby) along with its inputs. 

These tasks will be executed in parallel using Joblib to evaluate the performance of various data processing operations.

In [46]:
joblib_benchmarks = {
    'duration': [],  # in seconds
    'task': [],
}

joblib_benchmarks_filtered = {
    'duration': [],  # in seconds
    'task': [],
}

joblib_benchmarks_cache = {
    'duration': [],  # in seconds
    'task': [],
}

In [ ]:
# === Define the list of tasks to run in parallel using Joblib ===
tasks = [
    (read_file_parquet, None, {}, "read file"),
    (count, joblib_data, {}, "count"),
    (count_index_length, joblib_data, {}, "count index length"),
    (mean, joblib_data, {}, "mean"),
    (standard_deviation, joblib_data, {}, "standard deviation"),
    (mean_of_sum, joblib_data, {}, "mean of columns addition"),
    (sum_columns, joblib_data, {}, "addition of columns"),
    (mean_of_product, joblib_data, {}, "mean of columns multiplication"),
    (product_columns, joblib_data, {}, "multiplication of columns"),
    (value_counts, joblib_data, {}, "value counts"),
    (complicated_arithmetic_operation, joblib_data, {}, "complex arithmetic ops"),
    (mean_of_complicated_arithmetic_operation, joblib_data, {}, "mean of complex arithmetic ops"),
    (groupby_statistics, joblib_data, {}, "groupby statistics"),
    (join_count, joblib_data, {'other': other}, "join count"),
    (join_data, joblib_data, {'other': other}, "join"),
]

In [48]:
df_results = benchmark_joblib(tasks, n_jobs=-1, save_path='Results_Joblib/joblib_local_small')
print(df_results)

O arquivo Parquet foi salvo em Results_Joblib/joblib_local_small.
                                duration
task                                    
read file                       0.530811
count                           0.000009
count index length              0.000007
mean                            0.002038
standard deviation              0.008073
mean of columns addition        0.005543
addition of columns             0.002884
mean of columns multiplication  0.004239
multiplication of columns       0.002219
value counts                    0.006396
complex arithmetic ops          0.063138
mean of complex arithmetic ops  0.066550
groupby statistics              0.024920
join count                      0.009127
join                            0.008812


# cProfile

The function benchmark_cprofile benchmarks and profiles the execution of data processing functions using Python’s cProfile.

It collects both execution time and profiling statistics, then runs it for a set of predefined tasks. Finally, it aggregates all profiling statistics into a single DataFrame for further analysis or visualization.

In [ ]:
def benchmark_cprofile(f, df, benchmarks, name, profile_data=None, **kwargs):
    """
    Benchmark and profile the execution of a function using cProfile (without I/O).
    
    Parameters:
        f (callable): The function to benchmark.
        df (DataFrame): The DataFrame or data to pass as input.
        benchmarks (dict): Dictionary to store duration and task name.
        name (str): Name of the task/function.
        profile_data (list): List to store detailed profiling statistics.
        **kwargs: Additional arguments passed to the function.
    """ 
    
    pr = cProfile.Profile()
    pr.enable()

    start_time = time.perf_counter()
    ret = f(df, **kwargs)
    duration = time.perf_counter() - start_time

    pr.disable()

    benchmarks['duration'].append(duration)
    benchmarks['task'].append(name)
    print(f"{name} took: {duration:.4f} seconds")

    # Display top 15 most time-consuming function calls
    ps = pstats.Stats(pr)
    ps.sort_stats(pstats.SortKey.CUMULATIVE)
    ps.print_stats(15)

    # Save detailed profiling statistics into a list of DataFrames
    if profile_data is not None:
        stats_data = []
        for func, stat in ps.stats.items():
            filename, lineno, function = func
            ncalls, _, percall, cumtime, _ = stat
            stats_data.append({
                'task': name,
                'function': f"{filename}:{lineno} ({function})",
                'percall': percall,
                'cumtime': cumtime,
                'ncalls': ncalls
            })
        profile_data.append(pd.DataFrame(stats_data))

    return duration

In [ ]:
joblib_benchmarks_cprofile = {'duration': [], 'task': []}
joblib_profile_data = []

# === Execute benchmarks with cProfile for all tasks ===
benchmark_cprofile(read_file_parquet, df=None, benchmarks=joblib_benchmarks_cprofile, name='read file', profile_data=joblib_profile_data)
benchmark_cprofile(count, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='count', profile_data=joblib_profile_data)
benchmark_cprofile(count_index_length, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='count index length', profile_data=joblib_profile_data)
benchmark_cprofile(mean, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='mean', profile_data=joblib_profile_data)
benchmark_cprofile(standard_deviation, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='standard deviation', profile_data=joblib_profile_data)
benchmark_cprofile(mean_of_sum, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='mean of columns addition', profile_data=joblib_profile_data)
benchmark_cprofile(sum_columns, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='addition of columns', profile_data=joblib_profile_data)
benchmark_cprofile(mean_of_product, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='mean of columns multiplication', profile_data=joblib_profile_data)
benchmark_cprofile(product_columns, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='multiplication of columns', profile_data=joblib_profile_data)
benchmark_cprofile(value_counts, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='value counts', profile_data=joblib_profile_data)
benchmark_cprofile(complicated_arithmetic_operation, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='complex arithmetic ops', profile_data=joblib_profile_data)
benchmark_cprofile(mean_of_complicated_arithmetic_operation, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='mean of complex arithmetic ops', profile_data=joblib_profile_data)
benchmark_cprofile(groupby_statistics, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='groupby statistics', profile_data=joblib_profile_data)
benchmark_cprofile(join_count, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='join count', other=other, profile_data=joblib_profile_data)
benchmark_cprofile(join_data, df=joblib_data, benchmarks=joblib_benchmarks_cprofile, name='join', other=other, profile_data=joblib_profile_data)

read file took: 0.4357 seconds
         1787 function calls (1776 primitive calls) in 0.436 seconds

   Ordered by: cumulative time
   List reduced from 354 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.436    0.436 /var/tmp/ipykernel_10132/4001995804.py:6(read_file_parquet)
        1    0.000    0.000    0.436    0.436 /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:433(read_parquet)
        1    0.001    0.001    0.436    0.436 /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:201(read)
        1    0.000    0.000    0.241    0.241 {method 'to_pandas' of 'pyarrow.lib._PandasConvertible' objects}
        1    0.000    0.000    0.241    0.241 /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/pandas_compat.py:756(table_to_blockmanager)
        1    0.000    0.000    0.240    0.240 /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/pandas_compat

0.010022240000125748

In [ ]:
# Concatenate all DataFrames
df_profile_all = pd.concat(joblib_profile_data, ignore_index=True)

Once again, The print_profile_per_task function is used to print the slowest functions for each task that was profiled using cProfile.

Each task (like 'mean', 'join', etc.) has a corresponding DataFrame in the profile_data_list, which contains profiling information (e.g. time per function call and total cumulative time).

The function works as follows:

- Iterates over each DataFrame in the list (one per task).
- For each task, it sorts the profiling stats by the selected metric (percall or cumtime).
- Prints the top N slowest functions according to the selected metric.
- Displays the task name and a table with the function name and the chosen metric.

In [ ]:
def print_profile_per_task(profile_data_list, metric='percall', top_n=10):

    for df in profile_data_list:
        task_name = df['task'].iloc[0]  
        df_sorted = df.sort_values(by=metric, ascending=False).head(top_n)

        print(f"\n{'=' * 60}")
        print(f"Tarefa: {task_name}")
        print(f"Top {top_n} funções por '{metric}':\n")
        print(df_sorted[['function', metric]].to_string(index=False))


In [53]:
print_profile_per_task(joblib_profile_data, metric='percall', top_n=25)


Tarefa: read file
Top 25 funções por 'percall':

                                                                                                 function  percall
                                                                      ~:0 (<pyarrow.lib.table_to_blocks>) 0.239179
                                          ~:0 (<method 'to_table' of 'pyarrow._dataset.Dataset' objects>) 0.191227
                      /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/parquet.py:1377 (__init__) 0.001800
                         /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:201 (read) 0.000722
                                                              ~:0 (<built-in method builtins.isinstance>) 0.000106
                          /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/parquet.py:1440 (read) 0.000091
             /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/core/indexes/multi.py:1073 (_engine) 0.000073
                              

**Operations with filtering**

In [ ]:
print(f"Tipo do DataFrame : {type(joblib_data)}")

# Apply filter using query 
expr_filter = 'Tip_Amt >= 1 and Tip_Amt <= 5'
joblib_filtered = joblib_data.query(expr_filter)

print(f"Tipo do DataFrame filtrado: {type(joblib_filtered)}")

Tipo do DataFrame : <class 'pandas.core.frame.DataFrame'>
Tipo do DataFrame filtrado: <class 'pandas.core.frame.DataFrame'>


In [ ]:
tasks_filtered = [
    (read_file_parquet, None, {}, "read file"),
    (count, joblib_filtered, {}, "count"),
    (count_index_length, joblib_filtered, {}, "count index length"),
    (mean, joblib_filtered, {}, "mean"),
    (standard_deviation, joblib_filtered, {}, "standard deviation"),
    (mean_of_sum, joblib_filtered, {}, "mean of columns addition"),
    (sum_columns, joblib_filtered, {}, "addition of columns"),
    (mean_of_product, joblib_filtered, {}, "mean of columns multiplication"),
    (product_columns, joblib_filtered, {}, "multiplication of columns"),
    (value_counts, joblib_filtered, {}, "value counts"),
    (complicated_arithmetic_operation, joblib_filtered, {}, "complex arithmetic ops"),
    (mean_of_complicated_arithmetic_operation, joblib_filtered, {}, "mean of complex arithmetic ops"),
    (groupby_statistics, joblib_filtered, {}, "groupby statistics"),
    (join_count, joblib_filtered, {'other': other}, "join count"),
    (join_data, joblib_filtered, {'other': other}, "join"),
]

**Save results**

In [56]:
df_results_filtered = benchmark_joblib(tasks_filtered, n_jobs=-1, save_path='Results_Joblib/joblib_filtered_local_small')
print(df_results_filtered)

O arquivo Parquet foi salvo em Results_Joblib/joblib_filtered_local_small.
                                duration
task                                    
read file                       0.520832
count                           0.000005
count index length              0.000003
mean                            0.000726
standard deviation              0.001775
mean of columns addition        0.001155
addition of columns             0.000856
mean of columns multiplication  0.001172
multiplication of columns       0.000921
value counts                    0.002188
complex arithmetic ops          0.016577
mean of complex arithmetic ops  0.017529
groupby statistics              0.008633
join count                      0.007281
join                            0.006952


**Filtered with cProfile**

After that, a filtered benchmark was also performed for Joblib by selecting only rows where Tip_Amt is between 1 and 5. 

This filtered DataFrame was then used to re-run the same set of tasks to evaluate Joblib's performance under common preprocessing conditions.

In [ ]:
joblib_benchmarks_filtered_cprofile = {'duration': [], 'task': []}
joblib_filtered_profile_data = []

In [ ]:
# Execute benchmarks with cProfile
benchmark_cprofile(read_file_parquet, df=None, benchmarks=joblib_benchmarks_filtered_cprofile, name='read file', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(count, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='count', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(count_index_length, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='count index length', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(mean, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='mean', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(standard_deviation, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='standard deviation', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(mean_of_sum, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='mean of columns addition', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(sum_columns, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='addition of columns', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(mean_of_product, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='mean of columns multiplication', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(product_columns, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='multiplication of columns', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(value_counts, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='value counts', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(complicated_arithmetic_operation, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='complex arithmetic ops', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(mean_of_complicated_arithmetic_operation, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='mean of complex arithmetic ops', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(groupby_statistics, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='groupby statistics', profile_data=joblib_filtered_profile_data)
benchmark_cprofile(join_count, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='join count', other=other, profile_data=joblib_filtered_profile_data)
benchmark_cprofile(join_data, df=joblib_filtered, benchmarks=joblib_benchmarks_filtered_cprofile, name='join', other=other, profile_data=joblib_filtered_profile_data)

read file took: 0.4244 seconds
         1787 function calls (1776 primitive calls) in 0.424 seconds

   Ordered by: cumulative time
   List reduced from 354 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.424    0.424 /var/tmp/ipykernel_10132/4001995804.py:6(read_file_parquet)
        1    0.000    0.000    0.424    0.424 /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:433(read_parquet)
        1    0.001    0.001    0.424    0.424 /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:201(read)
        1    0.000    0.000    0.238    0.238 {method 'to_pandas' of 'pyarrow.lib._PandasConvertible' objects}
        1    0.000    0.000    0.238    0.238 /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/pandas_compat.py:756(table_to_blockmanager)
        1    0.000    0.000    0.237    0.237 /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/pandas_compat

0.0038048000001253968

In [ ]:
#Concatenate results
df_filtered_profile_all = pd.concat(joblib_filtered_profile_data, ignore_index=True)

In [ ]:
print_profile_per_task(joblib_filtered_profile_data, metric='percall', top_n=25)


Tarefa: read file
Top 25 funções por 'percall':

                                                                                                  function  percall
                                                                       ~:0 (<pyarrow.lib.table_to_blocks>) 0.236102
                                           ~:0 (<method 'to_table' of 'pyarrow._dataset.Dataset' objects>) 0.183001
                       /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/parquet.py:1377 (__init__) 0.001586
                          /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:201 (read) 0.000883
                                                        ~:0 (<method 'astype' of 'numpy.ndarray' objects>) 0.000155
                                                               ~:0 (<built-in method builtins.isinstance>) 0.000100
              /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/core/indexes/multi.py:1073 (_engine) 0.000090
                      

**Operations with cache**

To simulate a realistic scenario where filtered data is reused across tasks (e.g., in iterative workflows or pipelines), the  joblib.Memory was used to cache intermediate results. 

For this process we:

- Set up a cache directory (joblib_cache) where joblib stores memoized outputs.

- Defined a function get_filtered_data(df) to wrap the already-filtered DataFrame.

- Decorated it with @memory.cache, so that if the same input is passed again, the cached version is reused, avoiding recomputation.

This simulates "cached data" behavior, often found in pipelines or repeated analytics tasks.

Then a list of parallel tasks was built using this cached DataFrame (joblib_cache) and benchmarked them using the same benchmark_joblib function.

In [ ]:
from joblib import Memory

# Setup a directory where joblib will store cached results
memory = Memory(location='joblib_cache', verbose=0)

# Wrap the filtering logic (or any intermediate result) in a cached function
@memory.cache
def get_filtered_data(df):
    return df               # In this case, df is already filtered, so we just return it

# Apply the cached function to the filtered DataFrame
joblib_cache = get_filtered_data(joblib_filtered)  

In [ ]:
# === Define task list using the cached data ===
tasks_cache = [
    (read_file_parquet, None, {}, "read file"),
    (count, joblib_cache, {}, "count"),
    (count_index_length, joblib_cache, {}, "count index length"),
    (mean, joblib_cache, {}, "mean"),
    (standard_deviation, joblib_cache, {}, "standard deviation"),
    (mean_of_sum, joblib_cache, {}, "mean of columns addition"),
    (sum_columns, joblib_cache, {}, "addition of columns"),
    (mean_of_product, joblib_cache, {}, "mean of columns multiplication"),
    (product_columns, joblib_cache, {}, "multiplication of columns"),
    (value_counts, joblib_cache, {}, "value counts"),
    (complicated_arithmetic_operation, joblib_cache, {}, "complex arithmetic ops"),
    (mean_of_complicated_arithmetic_operation, joblib_cache, {}, "mean of complex arithmetic ops"),
    (groupby_statistics, joblib_cache, {}, "groupby statistics"),
    (join_count, joblib_cache, {'other': other}, "join count"),
    (join_data, joblib_cache, {'other': other}, "join"),
]

In [ ]:
# Run the Joblib benchmark using cached data (simulating in-memory reuse)
df_results_cache = benchmark_joblib(tasks_cache, n_jobs=-1, save_path='Results_Joblib/joblib_cache_local_small')
print(df_results_cache)

O arquivo Parquet foi salvo em Results_Joblib/joblib_cache_local_small.
                                duration
task                                    
read file                       0.480542
count                           0.000004
count index length              0.000003
mean                            0.000683
standard deviation              0.001526
mean of columns addition        0.001151
addition of columns             0.000986
mean of columns multiplication  0.001181
multiplication of columns       0.001281
value counts                    0.003089
complex arithmetic ops          0.017869
mean of complex arithmetic ops  0.017434
groupby statistics              0.008700
join count                      0.007081
join                            0.007736


**Cache with cProfile**

In this part of the report, cProfile is used to benchmark the performance of several data operations on a cached version of the filtered DataFrame (joblib_cache). By profiling each function individually, it's possible to identify bottlenecks and analyze which tasks are the most computationally expensive in terms of time per call and cumulative execution time.

In [ ]:
# === Benchmarks with cProfile using cached filtered data ===
joblib_benchmarks_cache_cprofile = {'duration': [], 'task': []}
joblib_cache_profile_data = []

In [ ]:
# Execute benchmarks with cProfile
benchmark_cprofile(read_file_parquet, df=None, benchmarks=joblib_benchmarks_cache_cprofile, name='read file', profile_data=joblib_cache_profile_data)
benchmark_cprofile(count, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='count', profile_data=joblib_cache_profile_data)
benchmark_cprofile(count_index_length, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='count index length', profile_data=joblib_cache_profile_data)
benchmark_cprofile(mean, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='mean', profile_data=joblib_cache_profile_data)
benchmark_cprofile(standard_deviation, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='standard deviation', profile_data=joblib_cache_profile_data)
benchmark_cprofile(mean_of_sum, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='mean of columns addition', profile_data=joblib_cache_profile_data)
benchmark_cprofile(sum_columns, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='addition of columns', profile_data=joblib_cache_profile_data)
benchmark_cprofile(mean_of_product, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='mean of columns multiplication', profile_data=joblib_cache_profile_data)
benchmark_cprofile(product_columns, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='multiplication of columns', profile_data=joblib_cache_profile_data)
benchmark_cprofile(value_counts, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='value counts', profile_data=joblib_cache_profile_data)
benchmark_cprofile(complicated_arithmetic_operation, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='complex arithmetic ops', profile_data=joblib_cache_profile_data)
benchmark_cprofile(mean_of_complicated_arithmetic_operation, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='mean of complex arithmetic ops', profile_data=joblib_cache_profile_data)
benchmark_cprofile(groupby_statistics, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='groupby statistics', profile_data=joblib_cache_profile_data)
benchmark_cprofile(join_count, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='join count', other=other, profile_data=joblib_cache_profile_data)
benchmark_cprofile(join_data, df=joblib_cache, benchmarks=joblib_benchmarks_cache_cprofile, name='join', other=other, profile_data=joblib_cache_profile_data)

read file took: 0.4663 seconds
         1787 function calls (1776 primitive calls) in 0.466 seconds

   Ordered by: cumulative time
   List reduced from 354 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.466    0.466 /var/tmp/ipykernel_10132/4001995804.py:6(read_file_parquet)
        1    0.000    0.000    0.466    0.466 /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:433(read_parquet)
        1    0.001    0.001    0.466    0.466 /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:201(read)
        1    0.000    0.000    0.264    0.264 {method 'to_pandas' of 'pyarrow.lib._PandasConvertible' objects}
        1    0.000    0.000    0.264    0.264 /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/pandas_compat.py:756(table_to_blockmanager)
        1    0.000    0.000    0.262    0.262 /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/pandas_compat

0.004705469999862544

In [ ]:
# Concatenate all results
df_cache_profile_all = pd.concat(joblib_cache_profile_data, ignore_index=True)

In [ ]:
print_profile_per_task(joblib_cache_profile_data, metric='percall', top_n=25)


Tarefa: read file
Top 25 funções por 'percall':

                                                                                                 function  percall
                                                                      ~:0 (<pyarrow.lib.table_to_blocks>) 0.261099
                                          ~:0 (<method 'to_table' of 'pyarrow._dataset.Dataset' objects>) 0.198923
                      /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/parquet.py:1377 (__init__) 0.002265
                         /opt/conda/envs/cdle/lib/python3.7/site-packages/pandas/io/parquet.py:201 (read) 0.000824
                                                              ~:0 (<built-in method builtins.isinstance>) 0.000116
                          /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/parquet.py:1440 (read) 0.000103
              /opt/conda/envs/cdle/lib/python3.7/site-packages/pyarrow/pandas_compat.py:1130 (<listcomp>) 0.000100
                              